In [ ]:

import json
from collections import Counter
from datetime import datetime, timedelta
from itertools import product, zip_longest
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as goa
import regex
import requests
import yaml
from plotly.colors import qualitative, sample_colorscale
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
from datetime import datetime
import pytz
from src.utils import (
    guardarExcel,
    guardarExcelMulti
)

from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
from src.utils import (
    isEmpty,
    loadEstaciones,
    loadLocalizaciones,
    localizeFecha,
    parallelizeFunction,
    rellenarId,
    removeDoubleQuotes,
    splitDataframe,
)
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isEmpty,
    map_cod2name,
    map_name2use_name,
    parallelizeFunction,
    setEF
)
from src.utils.util import (
    loadEstaciones,
    loadEstacionSinCTC
)
from src.processor import (
    XPECProcessor
    
)
from src.utils.topos import getEstacionamientos

In [ ]:
from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
# pd.set_option("future.no_silent_downcasting", True)
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

import argparse
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import regex
import yaml
from tqdm.auto import tqdm

from src.api import cargarHistorico, getHistoricoMOW
from src.processor import LogProcessor, XPECProcessor
from src.utils import isValidCode, parallelizeFunction, parseDate, rellenarId
from src.api.APIs import (
    getEstadoCirculacionesTecnicas,
    getPlanificacionCirculacionesTecnicas,
    getCirculacionesPlanificadas)

In [ ]:
# Tipos de tren que queremos
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "SUPRESIÓN",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ORIGEN",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    "Stopped": "STOP",
    "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "MANIOBRA_APROXIMACION"
            "MANIOBRA_LLEGADA",
            "MANIOBRA_SALIDA",
        ]
    )
}

In [ ]:
def cargarHistorico(
    start_date: str,
    end_date: str,
    estaciones: list[str],
    trenes: list[str],
    xSIV: bool = True,
    jCTC: bool = False,
    pro: bool = True,
    maniobra:bool = True
):
    # Comprobamos que la fecha de fin sea después de la de inicio
    if end_date <= start_date:
        end_date = (pd.to_datetime(start_date) + timedelta(days=1)).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    historico = getHistoricoMOW(
        estaciones=estaciones,
        trenes=trenes,
        inicio=start_date,
        fin=end_date,
        xSIV=xSIV,
        jCTC=jCTC,
        pro=pro,
        maniobra= maniobra
    )
    # historico = historico[
    #     (historico["Fecha"] >= pd.to_datetime(start_date))
    #     & (historico["Fecha"] <= pd.to_datetime(end_date))
    # ]
    # Usamos movimientos auditados
    # historico = historico[
    #     np.invert(historico["FuenteVía"].isin(["PLANNED", "SITRA_PROVIDED"]))
    # ]
    historico = historico[historico["NTécnico"].apply(isValidCode)].dropna(
        subset=["Movimiento"]
    )
    historico["mov_ord"] = historico["Movimiento"].apply(mov_sorter.get)
    return historico


In [ ]:
end_date = datetime.now().date().strftime("%Y-%m-%d") 
start_date = datetime.now().date()- timedelta(days=1)
start_date = start_date.strftime("%Y-%m-%d")
estaciones = []

In [ ]:
ntrenes = [rellenarId(el) for el in np.arange(100000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=True,
    jCTC=False,
    pro=True,
)
historico_pro = historico_pro.sort_values(
    by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
).reset_index(drop=True)

# Añadir información de la fecha
historico_pro["Día"] = historico_pro["Fecha"].dt.date
historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:

def leer_y_procesar_excel(archivo_path):
    """
    Lee un archivo Excel del formato LAV Madrid-Barcelona y completa las columnas.
    
    Args:
        archivo_path: Ruta al archivo Excel
    
    Returns:
        DataFrame procesado
    """
    
    # Leer el archivo Excel
    # Asumiendo que los datos comienzan en la fila 19 (índice 18)
    df = pd.read_excel(archivo_path, sheet_name='NORESTE MD-BCN-Frontera', 
                      skiprows=18, engine='openpyxl')
    
    print("Columnas encontradas:")
    print(df.columns.tolist())
    print("\nPrimeras filas:")
    print(df.head())
    
    return df


In [ ]:
archivo_path= Path(r"c:\Users\xiangzhou.zhang\Downloads\Revision Circulaciones AV.xlsx")
df = leer_y_procesar_excel(archivo_path)

In [ ]:
df

In [ ]:
import yaml

# Leer un archivo YAML
with open("data/Orden movimientos/Lineas AV.yaml", "r") as file:
    data = yaml.safe_load(file)


In [ ]:
# a = data["Madrid - Leon - Gijon"]

In [ ]:
# a_data = pd.DataFrame(a)

In [ ]:
# a_data.rename(columns={0:"Código"}, inplace=True)

In [ ]:
# a_data.head(2)

In [ ]:
# Leon_Asturia = historico_pro[historico_pro["Código"] == "17000"]

In [ ]:
# Leon_Asturia["LíneaComercial"].unique()

In [ ]:
C5_MAC = historico_pro[
    (historico_pro["LíneaComercial"] == "C5") &
    (historico_pro["CTC"] == "MAC")
].copy()

In [ ]:

def corregir_via(grupo):
    df = grupo.copy()
    mask = (df['Vía'].isna()) & (df['Movimiento'] == 'FIN')
    
    for idx in df[mask].index:
        secuencia = df.loc[idx, 'Secuencia']  # Ajusta el nombre de la columna de secuencia
        via_llegada = df[(df['Secuencia'] == secuencia) & (df['Movimiento'] == 'LLEGADA')]['Vía']
        if not via_llegada.empty:
            df.at[idx, 'Vía'] = via_llegada.iloc[0]
    
    return df

In [ ]:
def corregir_origenes(df_grupo):
    df_grupo = df_grupo.copy()
    if (df_grupo["Movimiento"] == "ORIGEN").sum() > 1:
        # conservar solo la última fila con "ORIGEN"
        idx_ult_origen = df_grupo[df_grupo["Movimiento"] == "ORIGEN"].index[-1]
        df_grupo = df_grupo.drop(df_grupo[(df_grupo["Movimiento"] == "ORIGEN") & (df_grupo.index != idx_ult_origen)].index)
    return df_grupo

In [ ]:
grupos = {}
for tecnico, grupo in C5_MAC.groupby('NTécnico'):
    grupo_corregido = corregir_origenes(grupo)
    grupo_corregido = corregir_via(grupo_corregido)
    grupos[tecnico] = grupo_corregido

In [ ]:
C5_MAC_modificado = pd.concat(grupos.values()).reset_index(drop=True)

In [ ]:
CódigoOrigen = "35607"
CódigoDestino = "35012"

In [ ]:
trayectoria = C5_MAC_modificado[(C5_MAC_modificado["CódigoOrigen"] == CódigoOrigen) & (C5_MAC_modificado["CódigoDestino"] == CódigoDestino)]

In [ ]:
trayectoria.reset_index(drop=True, inplace=True)

In [ ]:
grupos = dict(tuple(trayectoria.groupby('NTécnico')))

In [ ]:
primera_clave = list(grupos.keys())[0]
primer_df = grupos[primera_clave]

In [ ]:
df = primer_df.drop_duplicates(subset=['NTécnico',"Secuencia"])

In [ ]:
ruta = df[["Código","Nombre"]].copy()

In [ ]:
ruta.reset_index(drop= True, inplace=True)

In [ ]:
ruta_invertida = ruta.iloc[::-1].reset_index(drop=True)

In [ ]:
C5_MAC_modificado['Ntécnico_num'] = pd.to_numeric(C5_MAC_modificado['NTécnico'], errors='coerce')


df_par = C5_MAC_modificado[C5_MAC_modificado['Ntécnico_num'] % 2 == 0].copy()

df_impar = C5_MAC_modificado[C5_MAC_modificado['Ntécnico_num'] % 2 == 1].copy()

df_par.drop(columns='Ntécnico_num', inplace=True)
df_impar.drop(columns='Ntécnico_num', inplace=True)

In [ ]:
df_par["Vía"] = df_par["Vía"].fillna("DESCONICIDA")
df_impar["Vía"] = df_impar["Vía"].fillna("DESCONICIDA")

In [ ]:
Vías_par = df_par[["Código","Nombre", "Vía"]].drop_duplicates(keep="first")
Vías_impar = df_impar[["Código","Nombre", "Vía"]].drop_duplicates(keep="first")

In [ ]:
Vías_par.sort_values(by=["Código","Vía"], inplace=True)
Vías_par.reset_index(drop=True,inplace=True)
Vías_impar.sort_values(by=["Código","Vía"], inplace=True)
Vías_impar.reset_index(drop=True,inplace=True)

In [ ]:
# Vías["Vía"]= Vías["Vía"].fillna("DESCONOCIDA")

In [ ]:
recuento_vias_par = Vías_par.groupby(["Código","Nombre"]).size().reset_index(name='Nº total vías')
recuento_vias_impar = Vías_impar.groupby(["Código","Nombre"]).size().reset_index(name='Nº total vías')

In [ ]:
df_ida = pd.merge(ruta, recuento_vias_par, on=["Código","Nombre"], how="inner")

In [ ]:
df_vuelta = pd.merge(ruta, recuento_vias_impar, on=["Código","Nombre"], how="inner")

<h1> Nº circulaciónes reales y planificadas </h1>

In [ ]:
planificacion = getCirculacionesPlanificadas(start_date)

In [ ]:
planificacion

In [ ]:
codigo_ida = df_ida[["Código","Nombre"]].copy()

In [ ]:
planificado_idas = []
for codigo in codigo_ida["Código"].unique():
    planificado = planificacion[planificacion["Código"] == codigo].copy()
    planificado['Ntécnico_num'] = pd.to_numeric(planificado['NTécnico'], errors='coerce')
    planificado_par = planificado[planificado['Ntécnico_num'].notna() & (planificado['Ntécnico_num'] % 2 == 0)].copy()
    planificado_par.drop(columns='Ntécnico_num', inplace=True)
    planificado_idas.append(planificado_par)
planificado_ida = pd.concat(planificado_idas, ignore_index=True)


In [ ]:
planificado_vueltas = []
for codigo in codigo_ida["Código"].unique():
    planificado = planificacion[planificacion["Código"] == codigo].copy()
    planificado['Ntécnico_num'] = pd.to_numeric(planificado['NTécnico'], errors='coerce')
    planificado_impar = planificado[planificado['Ntécnico_num'].notna() & (planificado['Ntécnico_num'] % 2 == 1)].copy()
    planificado_impar.drop(columns='Ntécnico_num', inplace=True)
    planificado_vueltas.append(planificado_impar)
planificado_vueltas = pd.concat(planificado_vueltas, ignore_index=True)

In [ ]:
planificado_ida = planificado_ida[planificado_ida["Línea"] == "C5"]

In [ ]:
planificado_vueltas = planificado_vueltas[planificado_vueltas["Línea"] == "C5"]

In [ ]:
cir_planificada_ida = planificado_ida.groupby(['Código', 'Vía_Planificada']).size().reset_index(name='Nº Circulaciones planificadas')

In [ ]:
cir_planificada_vuelta = planificado_vueltas.groupby(['Código', 'Vía_Planificada']).size().reset_index(name='Nº Circulaciones planificadas')

In [ ]:
test = df_par[df_par["Código"]== "18000"]

In [ ]:
test2 = test[test["Vía"] =="9"]

In [ ]:
test2["FuenteVía"].unique()

In [ ]:
ida = df_par[(df_par["Movimiento"] == "LLEGADA") | (df_par["Movimiento"] == "ORIGEN")].copy()
# ida.drop_duplicates(subset=['NTécnico',"Código","Secuencia"], keep= "first",inplace=True)
# ida = ida[ida["FuenteVía"] == "CTC"].copy()

In [ ]:
vuelta = df_impar[(df_impar["Movimiento"] == "LLEGADA") | (df_impar["Movimiento"] == "ORIGEN")].copy()
vuelta.drop_duplicates(subset=['NTécnico',"Código","Secuencia"], keep= "first",inplace=True)
vuelta = vuelta[vuelta["FuenteVía"] == "CTC"].copy()

In [ ]:
Desconocida_ida = df_par[df_par["Vía"]=="DESCONICIDA"].copy()
Desconocida_vuelta = df_impar[df_impar["Vía"]=="DESCONICIDA"].copy()

In [ ]:
Desconocida_ida.drop_duplicates(subset=['NTécnico',"Código","Secuencia"], keep= "first",inplace=True)
Desconocida_vuelta.drop_duplicates(subset=['NTécnico',"Código","Secuencia"], keep= "first",inplace=True)

In [ ]:
circulación_deconocida_ida = Desconocida_ida.groupby(["Código","Vía"]).size().reset_index(name='Nº Circulaciones reales')
circulación_deconocida_vuelta = Desconocida_vuelta.groupby(["Código","Vía"]).size().reset_index(name='Nº Circulaciones reales')

In [ ]:
circulacion_real_ida = ida.groupby(["Código","Vía"]).size().reset_index(name='Nº Circulaciones reales')
circulacion_real_vuelta = vuelta.groupby(["Código","Vía"]).size().reset_index(name='Nº Circulaciones reales')

In [ ]:
circulacion_real_ida_1  = pd.concat([circulacion_real_ida, circulación_deconocida_ida], axis=0).reset_index(drop=True)
circulacion_real_vuelta_1  = pd.concat([circulacion_real_vuelta, circulación_deconocida_vuelta], axis=0).reset_index(drop=True)

In [ ]:
df_circulación_ida = pd.merge(
    cir_planificada_ida,
    circulacion_real_ida_1,
    left_on=["Código", "Vía_Planificada"],
    right_on=["Código", "Vía"],
    how="outer",
    suffixes=("_planificada", "_real")
)

In [ ]:
df_circulación_vuelta = pd.merge(
    cir_planificada_vuelta,
    circulacion_real_vuelta_1,
    left_on=["Código", "Vía_Planificada"],
    right_on=["Código", "Vía"],
    how="outer",
    suffixes=("_planificada", "_real")
)

In [ ]:
df_ida_1 = pd.merge(df_ida, df_circulación_ida,on= ["Código"], how="left")

In [ ]:
df_vuelta_1 = pd.merge(df_vuelta, df_circulación_vuelta,on= ["Código"], how="left")

In [ ]:
df_ida_1.head(6)

In [ ]:
Mostoles = df_par[df_par["Código"] == "35606"]

In [ ]:
Mostoles_planif = planificado_ida[planificado_ida["Código"] =="35607"]

In [ ]:
MostoleS_soto = df_par[df_par["Código"] == "35607"]

In [ ]:
test = MostoleS_soto[~MostoleS_soto["NTécnico"].isin(Mostoles["NTécnico"])]

In [ ]:
test["NTécnico"].unique()

In [ ]:
MostoleS_soto[~MostoleS_soto["NTécnico"].isin(Mostoles_planif["NTécnico"])]

In [ ]:
Mostoles_planif[~Mostoles_planif["NTécnico"].isin(MostoleS_soto["NTécnico"])]

In [ ]:
Mostoles_planif['NTécnico'].nunique()

In [ ]:
MostoleS_soto["NTécnico"].nunique()

In [ ]:
Mostoles[Mostoles["FuenteVía"] == "SITRA_AUDITED"]

In [ ]:
'19586', '20714', '20806', '20834'

In [ ]:
Mostoles["NTécnico"].nunique()

In [ ]:
MostoleS_soto[MostoleS_soto["Vía"]=="DESCONICIDA"]

In [ ]:
MostoleS_soto[MostoleS_soto["NTécnico"] == "20806"]

<h1> Rotulación </h1>

In [ ]:
rotulación_ida = ida.groupby(["Código","Vía","Movimiento"]).size().reset_index(name='Nº Circulaciones rotulados')
rotulación_vuelta = vuelta.groupby(["Código","Vía","Movimiento"]).size().reset_index(name='Nº Circulaciones rotulados')

In [ ]:
rotulado_ida = rotulación_ida[rotulación_ida["Movimiento"]=="ORIGEN"].copy()
rotulado_vuelta = rotulación_vuelta[rotulación_vuelta["Movimiento"]=="ORIGEN"].copy()

In [ ]:
rotulado_ida.drop(columns ='Movimiento', inplace=True)
rotulación_vuelta.drop(columns ='Movimiento', inplace=True)

In [ ]:
df_circulación_ida_1 = pd.merge(
    df_circulación_ida,
    rotulado_ida,
    left_on=["Código", "Vía"],
    right_on=["Código", "Vía"],
    how="outer"
)

In [ ]:
df_circulación_vuelta_1 = pd.merge(
    df_circulación_vuelta,
    rotulado_vuelta,
    left_on=["Código", "Vía"],
    right_on=["Código", "Vía"],
    how="outer"
)

In [ ]:
df_circulación_ida_1['Nº Circulaciones rotulados'] = (
    df_circulación_ida_1['Nº Circulaciones rotulados'].fillna(0)
)

In [ ]:
df_circulación_vuelta_1["Nº Circulaciones rotulados"] = (
    df_circulación_vuelta_1['Nº Circulaciones rotulados'].fillna(0)
)

In [ ]:
df_circulación_ida_1

<h1> Suprimidos</h1>

In [ ]:
Supresion_ida = df_par[df_par["Movimiento"]=="ELIMINACIÓN"].copy()
supresion_vuelta = df_impar[df_impar["Movimiento"]=="ELIMINACIÓN"].copy()

In [ ]:
Supresion_ida.drop_duplicates(subset=['NTécnico',"Código","Secuencia"], keep= "first",inplace=True)
supresion_vuelta.drop_duplicates(subset=['NTécnico',"Código","Secuencia"], keep= "first",inplace=True)

In [ ]:
suprimidos_ida = Supresion_ida.groupby(["Código","Vía"]).size().reset_index(name='Nº Circulaciones suprimidos')
suprimido_vuelta = supresion_vuelta.groupby(["Código","Vía"]).size().reset_index(name='Nº Circulaciones suprimidos')

In [ ]:
suprimidos_ida

In [ ]:
df_circulación_ida_2 = pd.merge(
    df_circulación_ida_1,
    suprimidos_ida,
    left_on=["Código", "Vía"],
    right_on=["Código", "Vía"],
    how="outer"
)

In [ ]:
df_circulación_vuelta_2 = pd.merge(
    df_circulación_vuelta_1,
    suprimido_vuelta,
    left_on=["Código", "Vía"],
    right_on=["Código", "Vía"],
    how="outer"
)

In [ ]:
df_circulación_ida_2['Nº Circulaciones suprimidos'] = (
    df_circulación_ida_2['Nº Circulaciones suprimidos'].fillna(0)
)

In [ ]:
df_circulación_vuelta_2['Nº Circulaciones suprimidos'] = (
    df_circulación_vuelta_2['Nº Circulaciones suprimidos'].fillna(0)
)

<h1> Previsión de supresión </h1>

In [ ]:

Supresion_ida["Fecha"] = pd.to_datetime(Supresion_ida["Fecha"])
Supresion_ida["SalidaPlanificada"] = pd.to_datetime(Supresion_ida["SalidaPlanificada"])
supresion_vuelta["Fecha"] = pd.to_datetime(supresion_vuelta["Fecha"])
supresion_vuelta["SalidaPlanificada"] = pd.to_datetime(supresion_vuelta["SalidaPlanificada"])

In [ ]:
Supresion_ida["AnticipaciónSupresión"] = (Supresion_ida["SalidaPlanificada"] - Supresion_ida["Fecha"])
supresion_vuelta["AnticipaciónSupresión"] = (supresion_vuelta["SalidaPlanificada"] - supresion_vuelta["Fecha"]) 

In [ ]:
def td_to_hhmmss(td):
    if pd.isna(td):
        return None
    total_seconds = int(td.total_seconds())           
    sign = "-" if total_seconds < 0 else ""
    total_seconds = abs(total_seconds)
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    return f"{sign}{hours:02d}:{minutes:02d}:{seconds:02d}"

In [ ]:
Supresion_ida["AnticipaciónSupresión_1"] = Supresion_ida["AnticipaciónSupresión"].apply(td_to_hhmmss)   
supresion_vuelta["AnticipaciónSupresión_1"] = supresion_vuelta["AnticipaciónSupresión"].apply(td_to_hhmmss)

In [ ]:
Supresion_ida[Supresion_ida["AnticipaciónSupresión_1"].notna()]
supresion_vuelta[supresion_vuelta["AnticipaciónSupresión_1"].notna()]

In [ ]:
Supresion_ida["AnticipaciónSupresión_1"] = pd.to_timedelta(Supresion_ida["AnticipaciónSupresión_1"], errors="coerce")
supresion_vuelta["AnticipaciónSupresión_1"] = pd.to_timedelta(supresion_vuelta["AnticipaciónSupresión_1"], errors="coerce")
media_por_codigo_ida = (
    Supresion_ida.groupby("Código")["AnticipaciónSupresión_1"]
    .mean()
    .reset_index()
)
media_por_codigo_vuelta = (
    supresion_vuelta.groupby("Código")["AnticipaciónSupresión_1"]
    .mean()
    .reset_index()
)

In [ ]:
media_por_codigo_ida["Previsión_Supresión(HH:mm:ss)"]=media_por_codigo_ida["AnticipaciónSupresión_1"].apply(td_to_hhmmss)  
media_por_codigo_vuelta["Previsión_Supresión(HH:mm:ss)"]= media_por_codigo_ida["AnticipaciónSupresión_1"].apply(td_to_hhmmss)

In [ ]:
media_por_codigo_vuelta

In [ ]:
media_por_codigo_ida.drop(columns="AnticipaciónSupresión_1", inplace=True)
media_por_codigo_vuelta.drop(columns="AnticipaciónSupresión_1", inplace=True)

In [ ]:
df_circulación_vuelta_3 = pd.merge(
    df_circulación_vuelta_2,
    media_por_codigo_vuelta,
    on =["Código"], how="left"
)
df_circulación_ida_3 = pd.merge(
    df_circulación_ida_2,
    media_por_codigo_ida,
    on =["Código"], how="left"
)

In [ ]:
df_circulación_ida_3.head(10)

<h1> Previsión vía </h1>

In [ ]:
ida_1 = df_par[
    (df_par["FuenteVía"] == "CTC") |
    ((df_par["FuenteVía"] == "UNKNOWN") & (df_par["Movimiento"] == "FIN"))
]

In [ ]:
ida_1 = ida_1[ida_1["FuenteVía"]=="CTC"]

In [ ]:
vuelta_1 = df_impar[
    (df_impar["FuenteVía"] == "CTC") |
    ((df_impar["FuenteVía"] == "UNKNOWN") & (df_impar["Movimiento"] == "FIN"))
]

In [ ]:
vuelta_1 = vuelta_1[vuelta_1["FuenteVía"] =="CTC"]

In [ ]:
def Calcular_anticipación (sub_dfs):
    información_adicional = []

    for tren in sub_dfs:
        if isinstance(tren, pd.DataFrame) and 'Movimiento' in tren.columns:
            mask_aproximacion = tren['Movimiento'].str.upper().isin(["APROXIMACIÓN", "PREVISIÓN"])
            mask_llegada = tren['Movimiento'].str.upper() == "LLEGADA"
            mask_origen = tren['Movimiento'].str.upper() == "ORIGEN"
            mask_salida = tren['Movimiento'].str.upper() == "SALIDA"
            mask_finalización = tren['Movimiento'].str.upper() == "FIN"

            aproximacion = tren[mask_aproximacion]
            llegada = tren[mask_llegada]
            origen = tren[mask_origen]
            salida = tren[mask_salida]
            fin = tren[mask_finalización]

            # === Caso ORIGEN ===
            if not origen.empty:
                tiempo_origen = origen["Fecha"].iloc[0]
                NTécnico = origen["NTécnico"].iloc[0]
                Código = origen["Código"].iloc[0]
                Fecha = origen["FechaOrigen"].iloc[0]
                Vía = origen["Vía"].iloc[0] if pd.notna(origen["Vía"].iloc[0]) and origen["Vía"].iloc[0] != "" else "NA"
                Elemento = origen["Elemento"].iloc[0] if pd.notna(origen["Elemento"].iloc[0]) and origen["Elemento"].iloc[0] != "" else "NA"
                Tipo_circulación = "ORIGEN"

                salida_despues = salida[salida["Fecha"] > tiempo_origen]
                if not salida_despues.empty:
                    tiempo_salida = salida_despues["Fecha"].iloc[0]
                    tiempo_anticipación_CTC = tiempo_salida - tiempo_origen
                else:
                    tiempo_anticipación_CTC = "NA"

                if tiempo_anticipación_CTC != "NA":
                    horas = tiempo_anticipación_CTC.total_seconds() // 3600
                    minutos = (tiempo_anticipación_CTC.total_seconds() % 3600) // 60
                    segundos = int(tiempo_anticipación_CTC.total_seconds() % 60)
                    tiempo_anticipación_CTC_formateado = f"{int(horas):02}:{int(minutos):02}:{segundos:02}"
                else:
                    tiempo_anticipación_CTC_formateado = "NA"

                column = {
                    "NTécnico": NTécnico,
                    "FechaOrigen": Fecha,
                    "Código": Código,
                    "Vía": Vía,
                    "Elemento": Elemento,
                    "Tiempo de anticipación CTC": tiempo_anticipación_CTC_formateado,
                    "Tipo circulación": Tipo_circulación
                }
                información_adicional.append(column)

            # === Caso APROXIMACIÓN o PREVISIÓN ===
            elif not aproximacion.empty:
                tiempo_aprox = aproximacion["Fecha"].iloc[0]
                NTécnico = aproximacion["NTécnico"].iloc[0]
                Código = aproximacion["Código"].iloc[0]
                Fecha = aproximacion["FechaOrigen"].iloc[0]
                Vía = aproximacion["Vía"].iloc[0] if pd.notna(aproximacion["Vía"].iloc[0]) and aproximacion["Vía"].iloc[0] != "" else "NA"
                Elemento = aproximacion["Elemento"].iloc[0] if pd.notna(aproximacion["Elemento"].iloc[0]) and aproximacion["Elemento"].iloc[0] != "" else "NA"

                llegada_despues = llegada[llegada["Fecha"] >= tiempo_aprox]
                if not llegada_despues.empty:
                    tiempo_llegada = llegada_despues["Fecha"].iloc[0]
                    tiempo_anticipación_CTC = tiempo_llegada - tiempo_aprox
                    Tipo_circulación = "FIN" if not fin.empty else "PASO"
                else:
                    tiempo_anticipación_CTC = "NA"
                    Tipo_circulación = "INCOMPLETA"

                if tiempo_anticipación_CTC != "NA":
                    horas = tiempo_anticipación_CTC.total_seconds() // 3600
                    minutos = (tiempo_anticipación_CTC.total_seconds() % 3600) // 60
                    segundos = int(tiempo_anticipación_CTC.total_seconds() % 60)
                    tiempo_anticipación_CTC_formateado = f"{int(horas):02}:{int(minutos):02}:{segundos:02}"
                else:
                    tiempo_anticipación_CTC_formateado = "NA"

                column = {
                    "NTécnico": NTécnico,
                    "FechaOrigen": Fecha,
                    "Código": Código,
                    "Vía": Vía,
                    "Elemento": Elemento,
                    "Tiempo de anticipación CTC": tiempo_anticipación_CTC_formateado,
                    "Tipo circulación": Tipo_circulación
                }
                información_adicional.append(column)

            # === Caso SIN ANTICIPACIÓN ===
            elif aproximacion.empty and (
                (not llegada.empty and not salida.empty) or
                (not llegada.empty and not fin.empty)
            ):
                NTécnico = llegada["NTécnico"].iloc[0]
                Código = llegada["Código"].iloc[0]
                Fecha = llegada["FechaOrigen"].iloc[0]
                Vía = llegada["Vía"].iloc[0] if pd.notna(llegada["Vía"].iloc[0]) and llegada["Vía"].iloc[0] != "" else "NA"
                Elemento = llegada["Elemento"].iloc[0] if pd.notna(llegada["Elemento"].iloc[0]) and llegada["Elemento"].iloc[0] != "" else "NA"

                column = {
                    "NTécnico": NTécnico,
                    "FechaOrigen": Fecha,
                    "Código": Código,
                    "Vía": Vía,
                    "Elemento": Elemento,
                    "Tiempo de anticipación CTC": "NA",
                    "Tipo circulación": "SIN ANTICIPACIÓN"
                }
                información_adicional.append(column)

            # === Caso DESCONOCIDO ===
            else:
                if not tren.empty:
                    NTécnico = tren["NTécnico"].iloc[0]
                    Código = tren["Código"].iloc[0]
                    Fecha = tren["FechaOrigen"].iloc[0]
                    Vía = tren["Vía"].iloc[0] if pd.notna(tren["Vía"].iloc[0]) and tren["Vía"].iloc[0] != "" else "NA"
                    Elemento = tren["Elemento"].iloc[0] if pd.notna(tren["Elemento"].iloc[0]) and tren["Elemento"].iloc[0] != "" else "NA"
                else:
                    NTécnico = Código = Fecha = Vía = Elemento = "NA"

                column = {
                    "NTécnico": NTécnico,
                    "FechaOrigen": Fecha,
                    "Código": Código,
                    "Vía": Vía,
                    "Elemento": Elemento,
                    "Tiempo de anticipación CTC": "NA",
                    "Tipo circulación": "DESCONOCIDA"
                }
                información_adicional.append(column)
        else:
            print("El elemento no es un DataFrame o no tiene columna 'Movimiento'")
    return información_adicional


In [ ]:
sub_dfs = [group for _, group in ida_1.groupby(['NTécnico', 'Código'])]

In [ ]:
información_adicional_ida = Calcular_anticipación(sub_dfs)

In [ ]:
df_prevision_via_ida = pd.DataFrame(información_adicional_ida)

In [ ]:
sub_dfs = [group for _, group in vuelta_1.groupby(['NTécnico', 'Código'])]
información_adicional_vuelta=  Calcular_anticipación(sub_dfs)
df_prevision_via_vuelta = pd.DataFrame(información_adicional_vuelta)

In [ ]:
df_filtrado_ida = df_prevision_via_ida[df_prevision_via_ida["Tiempo de anticipación CTC"] != "NA"].copy()
df_filtrado_ida["Tiempo_segundos"] = pd.to_timedelta(df_filtrado_ida["Tiempo de anticipación CTC"], errors="coerce").dt.total_seconds()
df_filtrado_ida = df_filtrado_ida.dropna(subset=["Tiempo_segundos"])
promedios_ida = (
    df_filtrado_ida.groupby(["Código", "Vía"], dropna=False)["Tiempo_segundos"]
    .mean()
    .reset_index()
)
promedios_ida["PREV VIA CTC"] = pd.to_timedelta(promedios_ida["Tiempo_segundos"], unit="s")
promedios_ida["PREV VIA CTC"] = promedios_ida["PREV VIA CTC"].apply(
    lambda x: f"{int(x.total_seconds() // 3600):02}:{int((x.total_seconds() % 3600) // 60):02}:{int(x.total_seconds() % 60):02}"
)


In [ ]:
df_filtrado_vuelta= df_prevision_via_vuelta[df_prevision_via_vuelta["Tiempo de anticipación CTC"] != "NA"].copy()
df_filtrado_vuelta["Tiempo_segundos"] = pd.to_timedelta(df_filtrado_vuelta["Tiempo de anticipación CTC"], errors="coerce").dt.total_seconds()
df_filtrado_vuelta = df_filtrado_vuelta.dropna(subset=["Tiempo_segundos"])
promedios_vuelta = (
    df_filtrado_vuelta.groupby(["Código", "Vía"], dropna=False)["Tiempo_segundos"]
    .mean()
    .reset_index()
)
promedios_vuelta["PREV VIA CTC"] = pd.to_timedelta(promedios_vuelta["Tiempo_segundos"], unit="s")
promedios_vuelta["PREV VIA CTC"] = promedios_vuelta["PREV VIA CTC"].apply(
    lambda x: f"{int(x.total_seconds() // 3600):02}:{int((x.total_seconds() % 3600) // 60):02}:{int(x.total_seconds() % 60):02}"
)

In [ ]:
promedios_ida = promedios_ida.drop(columns="Tiempo_segundos")
promedios_vuelta = promedios_vuelta.drop(columns="Tiempo_segundos")

In [ ]:
df_circulación_ida_4 = pd.merge(
    df_circulación_ida_3,
    promedios_ida,
    on=["Código", "Vía"],
    how="left"
)


In [ ]:
df_circulación_vuelta_4 = pd.merge(
    df_circulación_vuelta_3,
    promedios_vuelta,
    on=["Código", "Vía"],
    how="left"
)

<h1> Retraso medio CTC </h1>

In [ ]:
df_retraso_ida = ida_1.copy()
df_retraso_vuelta = vuelta_1.copy()

In [ ]:
df_retraso_ida[df_retraso_ida["Código"] == "35606"]

In [ ]:
df_retraso_ida = df_retraso_ida.dropna(subset=["Retraso (segundos)"])
df_retraso_ida = df_retraso_ida[df_retraso_ida["Movimiento"] =="APROXIMACIÓN"].copy()
promedios_retraso_ida = (
    df_retraso_ida.groupby(["Código", "Vía"], dropna=False)["Retraso (segundos)"]
    .mean()
    .reset_index()
)
promedios_retraso_ida["RETRASO MEDIO CTC"] = pd.to_timedelta(promedios_retraso_ida["Retraso (segundos)"], unit="s")
promedios_retraso_ida["RETRASO MEDIO CTC"] = promedios_retraso_ida["RETRASO MEDIO CTC"].apply(
    lambda x: f"{int(x.total_seconds() // 3600):02}:{int((x.total_seconds() % 3600) // 60):02}:{int(x.total_seconds() % 60):02}"
)


In [ ]:
df_retraso_vuelta = df_retraso_vuelta.dropna(subset=["Retraso (segundos)"])
df_retraso_vuelta = df_retraso_vuelta[df_retraso_vuelta["Movimiento"] =="APROXIMACIÓN"].copy()
promedios_retraso_vuelta = (
    df_retraso_vuelta.groupby(["Código", "Vía"], dropna=False)["Retraso (segundos)"]
    .mean()
    .reset_index()
)
promedios_retraso_vuelta["RETRASO MEDIO CTC"] = pd.to_timedelta(promedios_retraso_vuelta["Retraso (segundos)"], unit="s")
promedios_retraso_vuelta["RETRASO MEDIO CTC"] = promedios_retraso_vuelta["RETRASO MEDIO CTC"].apply(
    lambda x: f"{int(x.total_seconds() // 3600):02}:{int((x.total_seconds() % 3600) // 60):02}:{int(x.total_seconds() % 60):02}"
)

In [ ]:
promedios_retraso_ida.drop(columns="Retraso (segundos)", inplace=True)
promedios_retraso_vuelta.drop(columns="Retraso (segundos)", inplace=True)

In [ ]:
df_circulación_ida_5 = pd.merge(
    df_circulación_ida_4,
    promedios_retraso_ida,
    on=["Código", "Vía"],
    how="left",
)

In [ ]:
df_circulación_vuelta_5 = pd.merge(
    df_circulación_vuelta_4,
    promedios_retraso_vuelta,
    on=["Código", "Vía"],
    how="left",
)

In [ ]:
df_ida_1 = pd.merge(df_ida, df_circulación_ida_5,on= ["Código"], how="left")

In [ ]:
df_vuelta_1 = pd.merge(df_vuelta, df_circulación_vuelta_5,on= ["Código"], how="left")

In [ ]:
espacio = pd.DataFrame([[None]*5]*len(df_ida_1), columns=[f"espacio_{i}" for i in range(1,6)])

In [ ]:
df_completo = pd.concat([df_ida_1, df_vuelta_1], axis=1).reset_index(drop=True)

In [ ]:
df_concat = pd.concat([df_ida_1.reset_index(drop=True), espacio, df_vuelta_1.reset_index(drop=True)], axis=1)

In [ ]:
fruta= Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\ejemploLinea1.xlsx")
start_date = str(start_date)
df_ida_1["Código"] = df_ida_1["Código"].astype(int)
df_vuelta_1["Código"] = df_vuelta_1["Código"].astype(int)
data = {
    start_date+"_ida":df_ida_1,
    start_date+"_vuelta":df_vuelta_1,
}
guardarExcelMulti(data, fruta)

In [ ]:
guardarExcelMulti(data, fruta)

In [ ]:
df_ida_1[df_ida_1["Código"]== "18000"]